In [0]:
%sql
select 
`Row ID`,
`Order ID`,
`Order Date`,
`Ship Date`,
`Ship Mode`,
`Customer ID`,
`Customer Name`,
Segment,
Country,
City,
State,
`Postal Code`,
Region,
`Product ID`,
Category,
`Sub-Category`,
`Product Name`,
Sales,
Quantity,
Discount,
Profit
 FROM er_tech.superstore.tb_superstore

In [0]:
%sql
-- Dimensão produto
CREATE OR REPLACE TABLE er_tech.superstore.bronze_produto
USING DELTA
AS
SELECT DISTINCT
    `Product ID`   as product_id,
    `Product Name` as product_name,
    `Category`     as category,
    `Sub-Category` as sub_category,
    CURRENT_TIMESTAMP() AS load_date,
    'superstore_csv' AS source_system
FROM er_tech.superstore.tb_superstore
;

-- -- Dimensão customer
CREATE OR REPLACE TABLE er_tech.superstore.bronze_customer
USING DELTA
AS
SELECT DISTINCT
    `Customer ID`   as customer_id,
    `Customer Name` as customer_name,
    `Segment`       as segment,
    CURRENT_TIMESTAMP() AS load_date,
    'superstore_csv' AS source_system
FROM er_tech.superstore.tb_superstore
;

CREATE OR REPLACE TABLE er_tech.superstore.bronze_superstore
USING DELTA
AS
SELECT 
    `Row ID` as row_id,
    `Order ID` as order_id,
    `Order Date` as order_date,
    `Ship Date` as ship_date,
    `Ship Mode` as ship_mode,
    `Customer ID` as customer_id,
    `Product ID` as product_id, 
    `Sales` as vlr_sales,
    `Quantity` as quantity,
    `Discount` as vlr_discount,
    `Profit` as vlr_profit,
    CURRENT_TIMESTAMP() AS load_date,
    'superstore_csv' AS source_system
FROM er_tech.superstore.tb_superstore
;

## Test Layer

In [0]:
%sql
-- Teste realizado para verificar se numero do produto pode ser utilizado como chave.
with cte_product_adjust as (
    select 
        right(product_id, 8) as product_id_number
    from 
        er_tech.superstore.bronze_produto
)

select
    product_id_number,
    count(product_id_number)
from cte_product_adjust 
group by 
    product_id_number
having count(product_id_number) > 1
;

select * from er_tech.superstore.bronze_produto where product_id like '%10000240%';

In [0]:
%sql
select * from er_tech.superstore.bronze_customer where postal_code = '92024';

-- Teste postal code.
select 
    t.postal_code, 
    count(postal_code)
from 
    (
    select distinct
        postal_code, 
        city 
    from 
        er_tech.superstore.bronze_customer
    ) as t
group by 
    t.postal_code
having count(postal_code) > 1;


## Camada silver

In [0]:
%sql
CREATE OR REPLACE TABLE er_tech.superstore.silver_produto
    USING DELTA
    AS
        select 
            ROW_NUMBER() OVER (ORDER BY product_id) AS product_sk,
            regexp_replace(product_id, '[^a-zA-Z0-9]', '') as product_id,
            product_name,
            category,
            sub_category,
            CURRENT_TIMESTAMP() AS created_at,
            CURRENT_TIMESTAMP() AS updated_at
        from 
            er_tech.superstore.bronze_produto
        ;

In [0]:
%sql
SELECT DISTINCT 
    ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_sk,
    regexp_replace(customer_id, '[^a-zA-Z0-9]', '') as customer_id,
    customer_name,
    segment,
    load_date,
    source_system
FROM 
    er_tech.superstore.bronze_customer
;

In [0]:
%sql
select * from er_tech.superstore.silver_produto;

## Camada Gold

In [0]:
%sql
select 
    round(SUM(vlr_sales),2) as total_sales 
FROM 
    er_tech.superstore.bronze_superstore
;